# DuckDB veri ambarı

`02_sap_hazirlama.ipynb` tarafından oluşturulan CSV’leri okuyup ambarı hazırlıyor ve kontrol ediyor. Önce SAP hazırlama notebook’unu çalıştır.

DuckDB kurulu değilse ayrı bir hücrede `%pip install duckdb` çalıştır. Kurulum hücreleri mevcut tabloları korur; CSV değiştiğinde mevcut tabloyu otomatik yenilemez.

In [ ]:
import pandas as pd
import duckdb

# Kaynak veriyi karşılaştırma kontrolü için okuyor.
clean_df = pd.read_csv("../data/processed/MFG10YearTerminationData_clean.csv")

In [224]:
#Bağlantıyı oluşturuyoruz
con = duckdb.connect("../data/processed/warehouse.duckdb")

In [225]:
# Kişisel bilgiler CSV’sini veritabanına yüklüyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS raw_personal AS
    SELECT *
    FROM read_csv_auto('../data/processed/PA0002_personal_data.csv')
""")

In [226]:
# Organizasyon CSV’sini veritabanına yüklüyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS raw_org AS
    SELECT *
    FROM read_csv_auto('../data/processed/PA0001_org_assignment.csv')
""")

In [227]:
# Tablodaki satır sayısını kontrol ediyor.
con.execute("SELECT COUNT(*) FROM raw_org").fetchall()
#fetchall():Sorgunun sonucunu Python’a getirir.

[(49653,)]

In [228]:
# İşe giriş ve ayrılma olaylarını veritabanına yüklüyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS raw_actions AS
    SELECT *
    FROM read_csv_auto('../data/processed/PA0000_actions.csv')
""")

In [229]:
# Tablodaki satır sayısını kontrol ediyor.
con.execute("SELECT COUNT(*) FROM raw_actions").fetchall()

[(7769,)]

In [230]:
# Yıllık çalışan kayıtlarını veritabanına yüklüyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS raw_employee_year AS
    SELECT *
    FROM read_csv_auto('../data/processed/employee_year_metrics.csv')
""")

In [231]:
# Tablodaki satır sayısını kontrol ediyor.
con.execute("SELECT COUNT(*) FROM raw_employee_year").fetchall()

[(49653,)]

In [232]:
# Satır ve farklı çalışan sayılarını karşılaştırıyor.
con.execute("""
    SELECT
        COUNT(*) AS satir_sayisi,
        COUNT(DISTINCT PERNR) AS farkli_calisan
    FROM raw_personal
""").fetchdf()
#fetchdf(): SQL sorgusunun sonucunu pandas DataFrame olarak getirir.

,satir_sayisi,farkli_calisan
0,6284,6284


In [233]:
# Bağlantısı veya gerekli bilgisi eksik kayıtları sayıyor.
con.execute("""
    SELECT COUNT(*) AS eslesmeyen_kayit
    FROM raw_employee_year AS y
    LEFT JOIN raw_personal AS p
        ON y.EmployeeID = p.PERNR
    WHERE p.PERNR IS NULL
""").fetchdf()

,eslesmeyen_kayit
0,0


In [234]:
# Departmanları tekilleştirip bağlantı numarası veriyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS dim_department AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY department_name) AS department_key,
        department_name
    FROM (
        SELECT DISTINCT department_name
        FROM raw_employee_year
    )
""")

In [235]:
# Tablonun kayıtlarını sorgulayıp sonucu gösteriyor.
con.execute("SELECT * FROM dim_department").fetchdf()

,department_key,department_name
0,1,Accounting
1,2,Accounts Payable
2,3,Accounts Receiveable
3,4,Audit
4,5,Bakery
5,6,Compensation
6,7,Customer Service
7,8,Dairy
8,9,Employee Records
9,10,Executive


In [236]:
# Unvanları tekilleştirip bağlantı numarası veriyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS dim_job AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY job_title) AS job_key,
        job_title
    FROM (
        SELECT DISTINCT job_title
        FROM raw_employee_year
    )
""")

In [237]:
# Tablonun kayıtlarını sorgulayıp sonucu gösteriyor.
con.execute("SELECT * FROM dim_job").fetchdf()

,job_key,job_title
0,1,Accounting Clerk
1,2,Accounts Payable Clerk
2,3,Accounts Receiveable Clerk
3,4,Auditor
4,5,Baker
5,6,Bakery Manager
6,7,Benefits Admin
7,8,CEO
8,9,CHief Information Officer
9,10,Cashier


In [238]:
# Çalışanların kişisel bilgilerini ambar tablosuna alıyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS dim_employee AS
    SELECT
        PERNR AS employee_key,
        PERNR AS employee_id,
        GBDAT AS birth_date,
        GESCH AS gender_code
    FROM raw_personal
""")

In [239]:
# Tablodaki satır sayısını kontrol ediyor.
con.execute("SELECT COUNT(*) FROM dim_employee").fetchdf()

,count_star()
0,6284


In [240]:
# Mağaza, şehir ve iş birimi eşleşmelerine bağlantı numarası veriyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS dim_store AS
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY store_name, city_name, BUSINESS_UNIT
        ) AS store_key,
        store_name AS store_code,
        city_name,
        BUSINESS_UNIT AS business_unit
    FROM (
        SELECT DISTINCT store_name, city_name, BUSINESS_UNIT
        FROM raw_employee_year
    )
""")

In [241]:
# Tablonun kayıtlarını sorgulayıp sonucu gösteriyor.
con.execute("SELECT * FROM dim_store").fetchdf()

,store_key,store_code,city_name,business_unit
0,1,1,Abbotsford,STORES
1,2,2,Aldergrove,STORES
2,3,3,Bella Bella,STORES
3,4,4,Blue River,STORES
4,5,5,Burnaby,STORES
5,6,6,Chilliwack,STORES
6,7,7,Cortes Island,STORES
7,8,8,Cranbrook,STORES
8,9,9,Dawson Creek,STORES
9,10,10,Dease Lake,STORES


In [242]:
# Kayıt ve olay tarihlerini birleştirip yıl, ay, gün ve çeyreğini çıkarıyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS dim_date AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY date_value) AS date_key,
        date_value,
        YEAR(date_value) AS year,
        QUARTER(date_value) AS quarter,
        MONTH(date_value) AS month,
        DAY(date_value) AS day
    FROM (
        SELECT CAST(recorddate_key AS DATE) AS date_value
        FROM raw_employee_year

        UNION

        SELECT CAST(BEGDA AS DATE)
        FROM raw_actions
    )
    WHERE date_value IS NOT NULL
""")


In [243]:
# Tablonun kayıtlarını sorgulayıp sonucu gösteriyor.
con.execute("""
    SELECT * FROM dim_date
    ORDER BY date_value
    LIMIT 10
""").fetchdf()

,date_key,date_value,year,quarter,month,day
0,1,1989-08-28,1989,3,8,28
1,2,1989-08-31,1989,3,8,31
2,3,1989-09-02,1989,3,9,2
3,4,1989-09-05,1989,3,9,5
4,5,1989-09-08,1989,3,9,8
5,6,1989-09-09,1989,3,9,9
6,7,1989-09-10,1989,3,9,10
7,8,1989-09-15,1989,3,9,15
8,9,1989-09-16,1989,3,9,16
9,10,1989-09-20,1989,3,9,20


In [244]:
# Ayrılma sebebi ve türlerini tekilleştirip bağlantı numarası veriyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS dim_termination AS
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY termination_reason, termination_type
        ) AS termination_key,
        termination_reason,
        termination_type
    FROM (
        SELECT DISTINCT
            MASSG AS termination_reason,
            TERMTYPE AS termination_type
        FROM raw_actions
        WHERE CAST(MASSN AS INTEGER) = 10

        UNION

        SELECT 'Not Applicable', 'Not Applicable'
    )
""")

In [245]:
# Tablonun kayıtlarını sorgulayıp sonucu gösteriyor.
con.execute("SELECT * FROM dim_termination").fetchdf()


,termination_key,termination_reason,termination_type
0,1,Layoff,Involuntary
1,2,Not Applicable,Not Applicable
2,3,Resignation,Voluntary
3,4,Retirement,Voluntary


In [246]:
# Birden fazla ayrılma olayı bulunan çalışanları arıyor.
con.execute("""
    SELECT PERNR, COUNT(*) AS ayrilma_kaydi
    FROM raw_actions
    WHERE CAST(MASSN AS INTEGER) = 10
    GROUP BY PERNR
    HAVING COUNT(*) > 1
""").fetchdf()

,PERNR,ayrilma_kaydi


In [247]:
# Yıllık kayıtları çalışan, departman ve unvan tablolarına bağlıyor.
# Tablo zaten varsa mevcut veriyi koruyor.
con.execute("""
    CREATE TABLE IF NOT EXISTS fact_employee_year AS
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY y.EmployeeID, y.STATUS_YEAR, y.STATUS
        ) AS record_key,
        e.employee_key,
        d.department_key,
        j.job_key,
        y.STATUS_YEAR AS status_year,
        y.recorddate_key AS record_date,
        y.age,
        y.length_of_service,
        y.STATUS AS status,
        CASE WHEN y.STATUS = 'TERMINATED' THEN 1 ELSE 0 END AS left_flag,
        y.date_conflict_flag
    FROM raw_employee_year AS y
    LEFT JOIN dim_employee AS e
        ON y.EmployeeID = e.employee_id
    LEFT JOIN dim_department AS d
        ON y.department_name = d.department_name
    LEFT JOIN dim_job AS j
        ON y.job_title = j.job_title
""")

In [248]:
# Tablodaki satır sayısını kontrol ediyor.
con.execute("SELECT COUNT(*) FROM fact_employee_year").fetchdf()

,count_star()
0,49653


In [249]:
# Sütunları kontrol ediyor; gerekli sütunu yalnızca yoksa ekliyor.
existing_columns = con.execute("PRAGMA table_info('fact_employee_year')").fetchdf()["name"].tolist()
if "store_key" not in existing_columns:
    con.execute("""
        ALTER TABLE fact_employee_year
        ADD COLUMN store_key BIGINT
    """)


In [250]:
# Yıllık kayıtların mağaza ve iş birimi bağlantısını dolduruyor.
con.execute("""
    UPDATE fact_employee_year AS f
    SET store_key = s.store_key
    FROM raw_employee_year AS y
    JOIN dim_store AS s
        ON y.store_name = s.store_code
        AND y.city_name = s.city_name
        AND y.BUSINESS_UNIT = s.business_unit
    WHERE f.employee_key = y.EmployeeID
        AND f.status_year = y.STATUS_YEAR
        AND f.status = y.STATUS
""")

In [251]:
# Bağlantısı veya gerekli bilgisi eksik kayıtları sayıyor.
con.execute("""
    SELECT COUNT(*) AS eslesmeyen_kayit
    FROM fact_employee_year
    WHERE store_key IS NULL
""").fetchdf()

,eslesmeyen_kayit
0,0


In [252]:
# Sütunları kontrol ediyor; gerekli sütunu yalnızca yoksa ekliyor.
existing_columns = con.execute("PRAGMA table_info('fact_employee_year')").fetchdf()["name"].tolist()
if "date_key" not in existing_columns:
    con.execute("""
        ALTER TABLE fact_employee_year
        ADD COLUMN date_key BIGINT
    """)


In [253]:
# Kayıt tarihini tarih tablosuna bağlıyor.
con.execute("""
    UPDATE fact_employee_year AS f
    SET date_key = d.date_key
    FROM dim_date AS d
    WHERE CAST(f.record_date AS DATE) = d.date_value
""")

In [254]:
# Bağlantısı veya gerekli bilgisi eksik kayıtları sayıyor.
con.execute("""
    SELECT COUNT(*) AS eslesmeyen_tarih
    FROM fact_employee_year
    WHERE date_key IS NULL
""").fetchdf()

,eslesmeyen_tarih
0,0


In [255]:
# Sütunları kontrol ediyor; gerekli sütunu yalnızca yoksa ekliyor.
existing_columns = con.execute("PRAGMA table_info('fact_employee_year')").fetchdf()["name"].tolist()
if "termination_key" not in existing_columns:
    con.execute("""
        ALTER TABLE fact_employee_year
        ADD COLUMN termination_key BIGINT
    """)


In [256]:
# Aktif kayıtlara uygulanamaz ayrılma bilgisi bağlantısını veriyor.
con.execute("""
    UPDATE fact_employee_year
    SET termination_key = (
        SELECT termination_key
        FROM dim_termination
        WHERE termination_reason = 'Not Applicable'
          AND termination_type = 'Not Applicable'
    )
    WHERE left_flag = 0
""")

In [257]:
# Ayrılmış kayıtları ilgili ayrılma sebebi ve türüne bağlıyor.
con.execute("""
    UPDATE fact_employee_year AS f
    SET termination_key = t.termination_key
    FROM raw_actions AS a
    JOIN dim_termination AS t
        ON a.MASSG = t.termination_reason
        AND a.TERMTYPE = t.termination_type
    WHERE CAST(a.MASSN AS INTEGER) = 10
        AND f.employee_key = a.PERNR
        AND f.left_flag = 1
        AND f.status_year = YEAR(CAST(a.BEGDA AS DATE))
""")

In [258]:
# Bağlantısı veya gerekli bilgisi eksik kayıtları sayıyor.
con.execute("""
    SELECT COUNT(*) AS eslesmeyen_ayrilma_bilgisi
    FROM fact_employee_year
    WHERE termination_key IS NULL
""").fetchdf()

,eslesmeyen_ayrilma_bilgisi
0,0


In [259]:
# Sütunları kontrol ediyor; gerekli sütunu yalnızca yoksa ekliyor.
existing_columns = con.execute("PRAGMA table_info('fact_employee_year')").fetchdf()["name"].tolist()
if "termination_date" not in existing_columns:
    con.execute("""
        ALTER TABLE fact_employee_year
        ADD COLUMN termination_date DATE
    """)


In [260]:
# Ayrılmış kayıtlara gerçek ayrılma tarihini yazıyor.
con.execute("""
    UPDATE fact_employee_year AS f
    SET termination_date = CAST(a.BEGDA AS DATE)
    FROM raw_actions AS a
    WHERE CAST(a.MASSN AS INTEGER) = 10
        AND f.employee_key = a.PERNR
        AND f.left_flag = 1
        AND f.status_year = YEAR(CAST(a.BEGDA AS DATE))
""")

In [261]:
# Bağlantısı veya gerekli bilgisi eksik kayıtları sayıyor.
con.execute("""
    SELECT COUNT(*) AS tarihi_eksik_ayrilma
    FROM fact_employee_year
    WHERE left_flag = 1 AND termination_date IS NULL
""").fetchdf()

,tarihi_eksik_ayrilma
0,0


In [262]:
# Toplam kayıt, farklı çalışan, aktif ve ayrılma sayılarını kontrol ediyor.
con.execute("""
    SELECT
        COUNT(*) AS toplam_kayit,
        COUNT(DISTINCT employee_key) AS farkli_calisan,
        SUM(left_flag) AS ayrilma_kaydi,
        SUM(CASE WHEN status = 'ACTIVE' THEN 1 ELSE 0 END) AS aktif_kayit
    FROM fact_employee_year
""").fetchdf()

,toplam_kayit,farkli_calisan,ayrilma_kaydi,aktif_kayit
0,49653,6284,1485.0,48168.0


In [263]:
# Departmanlara göre ayrılan çalışan sayısını getiriyor.
con.execute("""
    SELECT
        d.department_name,
        COUNT(*) AS ayrilan_calisan
    FROM fact_employee_year AS f
    JOIN dim_department AS d
        ON f.department_key = d.department_key
    WHERE f.left_flag = 1
    GROUP BY d.department_name
    ORDER BY ayrilan_calisan DESC
""").fetchdf()

,department_name,ayrilan_calisan
0,Meats,377
1,Produce,353
2,Customer Service,262
3,Dairy,191
4,Bakery,112
5,Processed Foods,90
6,Store Management,31
7,HR Technology,9
8,Recruitment,8
9,Employee Records,6


In [264]:
# Ayrılma sebeplerine göre çalışan sayılarını getiriyor.
con.execute("""
    SELECT
        t.termination_reason,
        COUNT(*) AS ayrilan_calisan
    FROM fact_employee_year AS f
    JOIN dim_termination AS t
        ON f.termination_key = t.termination_key
    WHERE f.left_flag = 1
    GROUP BY t.termination_reason
    ORDER BY ayrilan_calisan DESC
""").fetchdf()

,termination_reason,ayrilan_calisan
0,Retirement,885
1,Resignation,385
2,Layoff,215


In [265]:
# Bağlantısı veya gerekli bilgisi eksik kayıtları sayıyor.
con.execute("""
    SELECT COUNT(*) AS eksik_baglanti
    FROM fact_employee_year
    WHERE employee_key IS NULL
       OR department_key IS NULL
       OR job_key IS NULL
       OR store_key IS NULL
       OR date_key IS NULL
       OR termination_key IS NULL
""").fetchdf()

,eksik_baglanti
0,0


In [266]:
# Aktif kayıtlara ayrılma bilgisi taşınmış mı kontrol ediyor.
con.execute("""
    SELECT COUNT(*) AS tutarsiz_aktif_kayit
    FROM fact_employee_year AS f
    JOIN dim_termination AS t
        ON f.termination_key = t.termination_key
    WHERE f.left_flag = 0
      AND (
          f.termination_date IS NOT NULL
          OR t.termination_reason <> 'Not Applicable'
          OR t.termination_type <> 'Not Applicable'
      )
""").fetchdf()

,tutarsiz_aktif_kayit
0,0


In [267]:
# Beş ek çalışan–yıl kaydını ve iki tarih uyarısını kontrol ediyor.
con.execute("""
    SELECT
        COUNT(*) - COUNT(DISTINCT (employee_key, status_year))
            AS ek_calisan_yil_kaydi,
        SUM(CASE WHEN date_conflict_flag THEN 1 ELSE 0 END)
            AS tarih_uyarili_kayit
    FROM fact_employee_year
""").fetchdf()

,ek_calisan_yil_kaydi,tarih_uyarili_kayit
0,5,2.0


In [268]:
# Mart 2014’te ayrılan çalışanları tarih, departman ve sebepleriyle getiriyor.
con.execute("""
    SELECT
        e.employee_id,
        f.termination_date,
        d.department_name,
        t.termination_reason
    FROM fact_employee_year AS f
    JOIN dim_employee AS e
        ON f.employee_key = e.employee_key
    JOIN dim_department AS d
        ON f.department_key = d.department_key
    JOIN dim_termination AS t
        ON f.termination_key = t.termination_key
    WHERE f.left_flag = 1
      AND f.termination_date >= DATE '2014-03-01'
      AND f.termination_date < DATE '2014-04-01'
    ORDER BY f.termination_date, e.employee_id
""").fetchdf()

,employee_id,termination_date,department_name,termination_reason
0,1346,2014-03-04,Employee Records,Retirement
1,1352,2014-03-11,Legal,Retirement
2,1359,2014-03-16,Meats,Retirement
3,1360,2014-03-17,Accounts Receiveable,Retirement
4,1362,2014-03-18,Meats,Retirement


In [269]:
dates = pd.to_datetime(clean_df["terminationdate_key"])

march_2014 = clean_df[
    (clean_df["STATUS"] == "TERMINATED")
    & (dates >= "2014-03-01")
    & (dates < "2014-04-01")
]

march_2014[[
    "EmployeeID",
    "terminationdate_key",
    "department_name",
    "termreason_desc"
]].sort_values(["terminationdate_key", "EmployeeID"])

,EmployeeID,terminationdate_key,department_name,termreason_desc
49129,1346,2014-03-04,Employee Records,Retirement
49245,1352,2014-03-11,Legal,Retirement
48640,1359,2014-03-16,Meats,Retirement
49126,1360,2014-03-17,Accounts Receiveable,Retirement
48501,1362,2014-03-18,Meats,Retirement


In [270]:
# Veritabanındaki değişiklikleri dosyaya yazdırıyor.
con.execute("CHECKPOINT")
#veritabanındaki tüm değişiklikleri diske yazmak için CHECKPOINT komutunu kullanıyoruz.
#Bu, veritabanının tutarlılığını sağlar ve olası veri kaybını önler.

In [ ]:
con.close() #veri tabanını kapatıyoruz